# Core 06 - Native Graph

Objetivo: construir y ejecutar un graph Unicamente mediante `toolkit.graph`. Este notebook fija el backend portable; LangGraph SDK se ensena por separado en frameworks/.

**Lugar en el modelo:** Graph expresa lógica y routing entre unidades; no es un Provider ni cambia dónde corre la inferencia.

**Evidencia exigida:** el estado final debe conservar el `RunResult` del nodo y el backend resuelto debe ser observable.

**Límite de la evidencia:** igualdad entre backends significa contrato de salida equivalente, no objetos nativos idénticos.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_GRAPH_ENGINE | portable | Cambiar explícitamente entre el backend portable y LangGraph. |
| AGENTIC_SYSTEMS_DEMO_SYMBOL | graph | Estado inicial que procesan los nodos. |
| route | evidencia del agente | Elegir la rama sin un loop manual alternativo. |

## 1) Modelo mental

```text
state - agent_node - normalized output - final state
```

No existe un loop alternativo en el notebook. La seleccion de backend pertenece a la libreria.

In [ ]:

import os

import agentic_systems as toolkit

GRAPH_ENGINE = os.getenv("AGENTIC_SYSTEMS_GRAPH_ENGINE", "portable")
SYMBOL_TO_INSPECT = "graph"
runtime = toolkit.runtime(provider="python-runtime")
system = toolkit.system(runtime=runtime)

toolkit.show_json({
    "requested_engine": GRAPH_ENGINE,
    "symbol": SYMBOL_TO_INSPECT,
    "runtime": runtime.describe(),
}, title="Graph inputs")

## 2) Tool y agent

La entrada es un simbolo elegido por configuracion. La respuesta se deriva de `toolkit.__all__`; no contiene una solucion precargada.

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "package_version": toolkit.__version__,
    }

inspector = system.agent(
    name="graph_public_api_inspector",
    instructions="Ejecuta la tool solicitada y conserva su evidencia.",
    tools=[inspect_public_api],
    runtime=runtime,
)

def inspect_input(state: dict) -> dict:
    return {"tool": "inspect_public_api", "input": {"symbol": state["symbol"]}}


def inspect_output(result, _state: dict) -> dict:
    return {
        "inspection_run": result,
        "inspection": toolkit.agent_output(result, kind="graph_node"),
    }

inspect_node = toolkit.agent_node(
    inspector,
    input=inspect_input,
    output=inspect_output,
    result_key=None,
    trace="inspection_trace",
    mode="eval",
)

## 3) Construir con `toolkit.graph`

El default `engine="portable"` es determinista. Puedes seleccionar `langgraph` explícitamente con `AGENTIC_SYSTEMS_GRAPH_ENGINE`; el notebook nunca cambia de backend silenciosamente.

In [ ]:
app = toolkit.graph(
    name="public_api_inspection_graph",
    engine=GRAPH_ENGINE,
    state=dict,
    nodes={"inspect": inspect_node},
    edges=[("START", "inspect"), ("inspect", "END")],
)

graph_inspection = app.inspect()
assert graph_inspection["kind"] in {"agentic-systems-native", "framework-native"}
toolkit.show_json(graph_inspection, title="Graph inspection")

## 4) Ejecutar y componer evidencia

El estado conserva el `RunResult` producido por el nodo. `compose_result` recibe ese resultado real; no se inventan engine, model, usage ni tool events.

In [ ]:
final_state = app.run({"symbol": SYMBOL_TO_INSPECT})
inspection_run = final_state["inspection_run"]
inspection = final_state["inspection"]
assert inspection_run.ok, inspection_run.errors
assert inspection["fields"]["symbol"] == SYMBOL_TO_INSPECT

result = toolkit.compose_result(
    text=f"Inspeccion publica completada para {SYMBOL_TO_INSPECT}.",
    data={"symbol": SYMBOL_TO_INSPECT, "inspection": inspection},
    results=[inspection_run],
    mode="graph",
    input={"symbol": SYMBOL_TO_INSPECT},
    meta={"graph_engine": app.engine},
)

assert result.ok, result.errors
toolkit.human_result(result, title="Graph RunResult", show_lineage=True)
toolkit.show_json(inspection, title="Salida del nodo")

## 5) Lineage y cobertura

Lineage se proyecta desde el estado final y la cobertura enumera llamadas realmente ejecutadas.

In [ ]:
lineage = app.lineage(
    final_state,
    question=f"Es {SYMBOL_TO_INSPECT} es publico?",
    goal="Conservar evidencia de un graph portable.",
    answer_keys=("inspection",),
)
toolkit.show(lineage, title="Graph Lineage")

api_coverage = [
    "toolkit.runtime",
    "toolkit.system",
    "toolkit.tool",
    "system.agent",
    "toolkit.agent_node",
    "toolkit.graph",
    "GraphApp.inspect",
    "GraphApp.run",
    "GraphApp.lineage",
    "toolkit.agent_output",
    "toolkit.compose_result",
    "toolkit.human_result",
    "toolkit.show",
    "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Graph API coverage")

## Resultado e interpretacion

La salida debe ser identica en contrato sin importar si `app.engine` reporta `langgraph` o `portable`.